# Generate Categories Data

Create SQL INSERT statements for the categories table and save to `category.insert.sql`.

## 1. Import Libraries

In [17]:
import os
import re
print("✅ Libraries imported")

✅ Libraries imported


## 2. Helper Functions

In [18]:
def slugify(text):
    """Convert Vietnamese text to URL-friendly slug"""
    text = text.strip().lower()
    from_chars = "àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ·/_,:;"
    to_chars = "aaaaaaaaaaaaaaaaaeeeeeeeeeeeiiiiiooooooooooooooooouuuuuuuuuuuyyyyyd------"
    for i in range(len(from_chars)):
        text = text.replace(from_chars[i], to_chars[i])
    text = re.sub(r'[^a-z0-9 -]', '', text)
    text = re.sub(r'\s+', '-', text)
    text = re.sub(r'-+', '-', text)
    return text.strip('-')

print("✅ Defined slugify function")

✅ Defined slugify function


## 3. Generate SQL file

In [19]:
# Define categories structure
categories = [
    {"name": "Thời Trang", "subcategories": [
        "Thời Trang Nam",
        "Thời Trang Nữ",
        "Giày Dép"
    ]},
    {"name": "Điện Tử", "subcategories": [
        "Điện Thoại & Máy Tính Bảng",
        "Laptop & PC",
        "Phụ Kiện Số",
        "Máy Ảnh & Quay Phim"
    ]},
    {"name": "Nghệ Thuật & Trang Sức", "subcategories": [
        "Tranh Ảnh",
        "Tượng & Điêu Khắc",
        "Trang Sức"
    ]}
]

# Define default image mapping for each category and subcategory
cat_images = {
    "Thời Trang": "https://images.unsplash.com/photo-1490481651871-ab68de25d43d?w=500",
    "Thời Trang Nam": "https://file.hstatic.net/1000284478/file/phong-cach-thoi-trang-nam-2024-b_26df758c91ef4fc78b185be0ff9f4cb0.jpg",
    "Thời Trang Nữ": "https://uvi.vn/wp-content/uploads/2022/03/Chan-vay-xoe-duoi-ca.jpg",
    "Giày Dép": "https://images.unsplash.com/photo-1549298916-b41d501d3772?w=500",
    "Điện Tử": "https://images.unsplash.com/photo-1498049794561-7780e7231661?w=500",
    "Điện Thoại & Máy Tính Bảng": "https://images.unsplash.com/photo-1511707171634-5f897ff02aa9?w=500",
    "Laptop & PC": "https://images.unsplash.com/photo-1588872657578-7efd1f1555ed?w=500",
    "Phụ Kiện Số": "https://storage.googleapis.com/48877118-7272-4a4d-b302-0465d8aa4548/f646d3c1-f6c3-4227-92ba-bd451d0401ea/8a9fa536-61df-4faa-9217-0033791bf9a5.jpg",
    "Máy Ảnh & Quay Phim": "https://images.unsplash.com/photo-1516035069371-29a1b244cc32?w=500",
    "Nghệ Thuật & Trang Sức": "https://images.unsplash.com/photo-1513519245088-0e12902e5a38?w=500",
    "Tranh Ảnh": "https://images.unsplash.com/photo-1579783902614-a3fb3927b6a5?w=500",
    "Tượng & Điêu Khắc": "https://images.unsplash.com/photo-1578301978693-85fa9c0320b9?w=500",
    "Trang Sức": "https://images.unsplash.com/photo-1599643478518-a784e5dc4c8f?w=500"
}

sql_statements = []
sql_statements.append("-- Reset categories table")
sql_statements.append("TRUNCATE TABLE categories RESTART IDENTITY CASCADE;\n")

current_id = 1
parent_ids = {}

sql_statements.append("-- Insert parent categories")
for category in categories:
    name = category["name"]
    slug = slugify(name)
    image = cat_images.get(name, "https://shop-images.imgix.net/default_parent.jpg")
    sql = f"INSERT INTO categories (id, name, parent_id, status, deleted, description, slug, cat_image) VALUES ({current_id}, '{name}', NULL, 'active', FALSE, 'Category {name}', '{slug}', '{image}');"
    sql_statements.append(sql)
    parent_ids[name] = current_id
    current_id += 1

sql_statements.append("\n-- Insert child subcategories")
for category in categories:
    parent_name = category["name"]
    parent_id = parent_ids[parent_name]
    
    for sub_name in category["subcategories"]:
        slug = slugify(sub_name)
        image = cat_images.get(sub_name, "https://shop-images.imgix.net/default_child.jpg")
        sql = f"INSERT INTO categories (id, name, parent_id, status, deleted, description, slug, cat_image) VALUES ({current_id}, '{sub_name}', {parent_id}, 'active', FALSE, '{sub_name} under {parent_name}', '{slug}', '{image}');"
        sql_statements.append(sql)
        current_id += 1

sql_statements.append(f"\n-- Adjust ID sequence")
sql_statements.append(f"SELECT setval('categories_id_seq', (SELECT MAX(id) FROM categories));")

# Write SQL statements to category.insert.sql
output_file = "category.insert.sql"
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements))

print(f"🎉 Successfully generated {output_file} with {len(categories) + sum(len(c['subcategories']) for c in categories)} categories!")

🎉 Successfully generated category.insert.sql with 13 categories!
